# PubMed 20k RCT — Sequential Sentence Classification

**Assignment 5 — Stochastic Models for CS**
Gutenberg Petit-vil

---

## 1. Introduction

The idea of this assignment is to take medical research abstracts (specifically
randomized controlled trials from PubMed) and figure out the role each sentence
plays in the abstract. Most of these abstracts kinda follow the same pattern —
some background info, the objective of the study, the methods they used, the
results, and then a conclusion at the end. So the goal here is to train a model
that can read a sentence and tell you which of those five categories it belongs
to.

This is useful because if you can do this automatically you can speed up a lot
of literature review work for doctors and researchers. There's a ton of
biomedical papers out there and nobody has time to read all of them.

A few things make this harder than a basic text classification problem:

- It's **sequential** — where the sentence sits in the abstract actually
  matters. An "Objective" sentence almost always shows up early, conclusions
  show up at the end, etc.
- Sentences are pretty different in length and the vocabulary is medical, so
  there's a lot of weird words.
- The classes aren't balanced. RESULTS and METHODS make up most of the data
  and OBJECTIVE / BACKGROUND are way smaller.

For this assignment we have to build and compare four different models:

1. **Simple RNN** — basic recurrent baseline, kind of the "starting point"
2. **LSTM** — better recurrent network with gates that handles longer sequences
3. **Transformer** — self-attention based, no recurrence
4. **BERT** — pre-trained transformer that we fine-tune on this task

The idea is to see how each one does and figure out which approach actually
works best for this kind of biomedical text.

> **How to run this:** open in Google Colab, switch the runtime to GPU
> (`Runtime → Change runtime type → GPU`), drag `train.csv`, `val.csv`, and
> `test.csv` into the file panel on the left, then `Runtime → Run all`. BERT
> takes a while so don't be surprised if it sits there for a bit.


## 2. Setup

Just installing the libraries we need and setting a seed so results are at
least somewhat reproducible. Most of this is already in Colab — only
`transformers` needs to be added.


In [ ]:
# Install transformers (Colab usually has torch + sklearn pre-installed)
!pip install -q transformers==4.41.2 --upgrade


In [ ]:
import os, re, math, time, random, json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 3. Dataset

### 3.1 What's in it

The dataset is **PubMed 20k RCT**, basically ~20k RCT abstracts from PubMed
where every sentence has been labeled with one of the five categories. We
got three CSVs (train, val, test) and each row is one sentence. The columns
are:

| column | what it is |
|--------|---------|
| `abstract_id` | PubMed ID — same id means the sentences are from the same abstract |
| `line_id` | unique id for the sentence |
| `abstract_text` | the actual sentence |
| `line_number` | which sentence it is in the abstract (starting at 0) |
| `total_lines` | how many sentences the abstract has total |
| `current_line` | just `"{line_number}_{total_lines}"` as a string |
| `target` | the label — BACKGROUND, OBJECTIVE, METHODS, RESULTS, or CONCLUSIONS |

### 3.2 Loading the files


In [ ]:
# Robust loader — accepts either "test.csv" or the Canvas filename "test (1).csv"
def first_existing(candidates):
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"None of these files exist: {candidates}")

train_path = first_existing(["train.csv"])
val_path   = first_existing(["val.csv", "dev.csv"])
test_path  = first_existing(["test.csv", "test (1).csv"])

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

print("loaded:", train_path, val_path, test_path)
print("shapes:", train_df.shape, val_df.shape, test_df.shape)
train_df.head()


### 3.3 Looking at the data first

In [ ]:
# Class distribution per split
classes_sorted = sorted(train_df["target"].unique())
print("classes:", classes_sorted)
print("\nClass counts (train):")
print(train_df["target"].value_counts())

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for i, (name, df) in enumerate([("train", train_df), ("val", val_df), ("test", test_df)]):
    df["target"].value_counts().reindex(classes_sorted).fillna(0).astype(int)\
        .plot(kind="bar", ax=ax[i], color="steelblue", edgecolor="white")
    ax[i].set_title(f"{name} ({len(df):,} sentences)")
    ax[i].set_ylabel("count")
    ax[i].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig("label_distribution.png", dpi=120); plt.show()


In [ ]:
# Sentence length statistics
lengths = train_df["abstract_text"].str.split().apply(len)
print(f"avg words/sentence: {lengths.mean():.2f}")
print(f"median: {int(lengths.median())}, p95: {int(lengths.quantile(0.95))}, max: {lengths.max()}")

plt.figure(figsize=(8, 4))
plt.hist(lengths.clip(upper=120), bins=60, color="darkorange", edgecolor="white")
plt.axvline(lengths.quantile(0.95), color="red", linestyle="--",
            label=f"p95 = {int(lengths.quantile(0.95))} words")
plt.title("Sentence length (words) — train")
plt.xlabel("words"); plt.ylabel("freq"); plt.legend()
plt.tight_layout(); plt.savefig("sentence_length_hist.png", dpi=120); plt.show()


In [ ]:
# Vocabulary peek
def quick_tokenize(s):
    return re.findall(r"\b\w+\b", s.lower())

vocab_counter = Counter()
for s in train_df["abstract_text"]:
    vocab_counter.update(quick_tokenize(s))
print(f"unique tokens (train): {len(vocab_counter):,}")
print("top 10:", vocab_counter.most_common(10))


**A few things from the plots above:**

- The classes are pretty unbalanced. RESULTS and METHODS together are like
  ~70% of the data, and OBJECTIVE / BACKGROUND are way smaller. The model
  is gonna have an easier time learning the big classes.
- The three splits look pretty similar though, which is good — means the
  test set is a fair check.
- Sentences are ~26 words on average and the 95th percentile is around 54
  words. So I picked `MAX_LEN = 64` to cover most of them without cutting
  much off.
- The training set has like ~36k unique words. I'm only keeping the top
  30k for the embedding models (plus a `<PAD>` and `<OOV>` token). BERT
  doesn't care because it has its own tokenizer.


## 4. Preprocessing

### 4.1 For the RNN / LSTM / Transformer

These three all use the same preprocessing because they all want integer
sequences. Steps are:

1. Lowercase everything and split into words with a regex.
2. Build a vocab from the **training data only** so we don't leak val/test
   words in.
3. Turn each sentence into a list of integers. Anything not in the vocab
   becomes `<OOV>` (index 1).
4. Pad or cut every sentence to length 64 so they're all the same shape.
5. Encode the labels (the five class names) into 0–4.
6. Throw it all in a `Dataset` + `DataLoader`.

### 4.2 For BERT

BERT has its own tokenizer (`bert-base-uncased`, WordPiece) so we just use
that. Max length 96 since sub-word tokens are smaller than full words. The
labels are still 0–4 like before.


In [ ]:
# ---- Hyperparameters that affect preprocessing ----
VOCAB_SIZE  = 30_000      # cap to keep embeddings manageable
MAX_LEN     = 64          # >= p95 of sentence lengths
BATCH_SIZE  = 64

# ---- Tokenize ----
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

train_tokens = [tokenize(t) for t in train_df["abstract_text"]]
val_tokens   = [tokenize(t) for t in val_df["abstract_text"]]
test_tokens  = [tokenize(t) for t in test_df["abstract_text"]]

# ---- Build vocab on TRAIN only (avoid leakage) ----
counter = Counter(tok for toks in train_tokens for tok in toks)
most_common = counter.most_common(VOCAB_SIZE - 2)
word2idx = {w: i + 2 for i, (w, _) in enumerate(most_common)}
word2idx["<PAD>"] = 0
word2idx["<OOV>"] = 1
PAD_IDX, OOV_IDX = 0, 1
print(f"vocab size: {len(word2idx):,}")

def encode(tokens):
    return [word2idx.get(t, OOV_IDX) for t in tokens][:MAX_LEN]

def encode_and_pad(token_lists):
    seqs = [torch.tensor(encode(t), dtype=torch.long) for t in token_lists]
    padded = pad_sequence(seqs, batch_first=True, padding_value=PAD_IDX)
    if padded.shape[1] < MAX_LEN:
        padded = torch.nn.functional.pad(padded, (0, MAX_LEN - padded.shape[1]), value=PAD_IDX)
    return padded[:, :MAX_LEN]

X_train = encode_and_pad(train_tokens)
X_val   = encode_and_pad(val_tokens)
X_test  = encode_and_pad(test_tokens)
print("X_train:", X_train.shape, " X_val:", X_val.shape, " X_test:", X_test.shape)

# ---- Label encoding ----
le = LabelEncoder()
le.fit(train_df["target"])
NUM_CLASSES = len(le.classes_)
print("classes:", list(le.classes_))

y_train = torch.tensor(le.transform(train_df["target"]), dtype=torch.long)
y_val   = torch.tensor(le.transform(val_df["target"]),   dtype=torch.long)
y_test  = torch.tensor(le.transform(test_df["target"]),  dtype=torch.long)


In [ ]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

train_loader = DataLoader(TextDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TextDataset(X_val, y_val),     batch_size=BATCH_SIZE)
test_loader  = DataLoader(TextDataset(X_test, y_test),   batch_size=BATCH_SIZE)


## 5. Training and Evaluation helpers

I'm reusing the same training loop and eval function for all four models so
the comparison is actually fair. Same optimizer (Adam), same loss
(cross-entropy), same batch size, same data. BERT gets its own loop later
because it needs AdamW + a scheduler + the HuggingFace input format.


In [ ]:
def train_model(model, train_loader, val_loader, *, epochs=3, lr=1e-3,
                weight_decay=0.0, log_every=200, name="model"):
    """Train and return per-epoch (train_loss, val_acc, val_f1) history."""
    model = model.to(device)
    optim_ = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    crit   = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_acc": [], "val_f1": []}

    for ep in range(1, epochs + 1):
        model.train()
        total = 0.0
        t0 = time.time()
        for step, (xb, yb) in enumerate(train_loader, 1):
            xb, yb = xb.to(device), yb.to(device)
            optim_.zero_grad()
            logits = model(xb)
            loss = crit(logits, yb)
            loss.backward()
            optim_.step()
            total += loss.item()
            if step % log_every == 0:
                print(f"[{name}] ep{ep} step{step} running loss {total/step:.4f}")

        # ---- val ----
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                p = model(xb).argmax(dim=1).cpu().numpy()
                preds.extend(p); trues.extend(yb.numpy())
        acc = accuracy_score(trues, preds)
        _, _, f1, _ = precision_recall_fscore_support(trues, preds, average="weighted", zero_division=0)
        history["train_loss"].append(total / max(len(train_loader), 1))
        history["val_acc"].append(acc)
        history["val_f1"].append(f1)
        print(f"[{name}] epoch {ep}/{epochs}  loss={history['train_loss'][-1]:.4f}  "
              f"val_acc={acc:.4f}  val_f1={f1:.4f}  ({time.time()-t0:.1f}s)")
    return history


def evaluate(model, loader, label_names):
    """Test-set evaluation. Prints classification report and returns metrics."""
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            p = model(xb).argmax(dim=1).cpu().numpy()
            preds.extend(p); trues.extend(yb.numpy())
    acc = accuracy_score(trues, preds)
    pr, rc, f1, _ = precision_recall_fscore_support(trues, preds, average="weighted", zero_division=0)
    print(classification_report(trues, preds, target_names=label_names, zero_division=0))
    return {"accuracy": acc, "precision": pr, "recall": rc, "f1": f1,
            "y_true": trues, "y_pred": preds}


## 6. Model A — Simple RNN

This is the most basic version. Just an embedding layer, a vanilla RNN, and
a linear layer for the prediction. We grab the final hidden state of the
RNN and use that as the sentence representation. It's the simplest model
here and honestly I'm not expecting it to do great — it's mostly a baseline
to see how much better the other ones are.

```
Embedding(V, 64)  →  RNN(64 → 128, tanh)  →  Dropout(0.3)  →  Linear(128 → 5)
```


In [ ]:
class SimpleRNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, num_classes=5, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, nonlinearity="tanh")
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        e = self.emb(x)                       # (B, T, E)
        _, h = self.rnn(e)                    # h: (1, B, H)
        h = h.squeeze(0)                      # (B, H)
        return self.fc(self.drop(h))

rnn_model = SimpleRNNClassifier(vocab_size=len(word2idx),
                                embed_dim=64, hidden_dim=128,
                                num_classes=NUM_CLASSES)
print(rnn_model)
rnn_history = train_model(rnn_model, train_loader, val_loader,
                          epochs=3, lr=1e-3, name="RNN")
rnn_results = evaluate(rnn_model, test_loader, list(le.classes_))


## 7. Model B — Bidirectional LSTM

LSTM is basically the better version of an RNN. The gates let it remember
stuff from way earlier in the sentence which the simple RNN can't really
do. I'm using a bidirectional one so it reads the sentence forward AND
backward, then concats both hidden states together. Should do noticeably
better than the plain RNN.

```
Embedding(V, 128)  →  BiLSTM(128 → 128)  →  concat(fwd_h, bwd_h)  →
   Linear(256 → 64) → ReLU → Dropout(0.3) → Linear(64 → 5)
```


In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128,
                 num_classes=5, dropout=0.3, num_layers=1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.drop = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_dim * 2, 64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        e = self.emb(x)
        _, (h, _) = self.lstm(e)              # h: (2*L, B, H)
        h_cat = torch.cat([h[-2], h[-1]], dim=1)  # (B, 2H)
        z = torch.relu(self.fc1(self.drop(h_cat)))
        return self.fc2(z)

lstm_model = LSTMClassifier(vocab_size=len(word2idx),
                            embed_dim=128, hidden_dim=128,
                            num_classes=NUM_CLASSES, dropout=0.3)
print(lstm_model)
lstm_history = train_model(lstm_model, train_loader, val_loader,
                           epochs=3, lr=1e-3, name="LSTM")
lstm_results = evaluate(lstm_model, test_loader, list(le.classes_))


## 8. Model C — Transformer (built from scratch)

Now instead of recurrence we're doing self-attention. I'm using PyTorch's
built-in `TransformerEncoderLayer` and stacking 2 of them with 4 heads.
There's also a positional embedding so the model knows the order of words
(otherwise attention is permutation-invariant). After the encoder I just
do a mean pool over the non-padding tokens and run that through a linear
head. The padding mask is passed in so attention doesn't waste time on
the `<PAD>` tokens.

```
TokenEmb(V, 128) + PosEmb(64, 128)
  →  TransformerEncoder × 2  (heads = 4, ff_dim = 256, GELU)
  →  mean-pool over non-padding positions
  →  Linear(128 → 64) → ReLU → Dropout(0.2) → Linear(64 → 5)
```


In [ ]:
class PositionalEmbedding(nn.Module):
    def __init__(self, max_len, dim):
        super().__init__()
        self.pos = nn.Embedding(max_len, dim)
    def forward(self, x):
        T = x.size(1)
        idx = torch.arange(T, device=x.device).unsqueeze(0)
        return x + self.pos(idx)


class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_heads=4, num_layers=2,
                 ff_dim=256, num_classes=5, dropout=0.2, max_len=MAX_LEN):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.pos_emb = PositionalEmbedding(max_len, embed_dim)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.fc1 = nn.Linear(embed_dim, 64)
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        mask = (x == PAD_IDX)                                   # (B, T)
        h = self.pos_emb(self.tok_emb(x))                       # (B, T, E)
        h = self.encoder(h, src_key_padding_mask=mask)
        # mean pool over non-pad tokens
        valid = (~mask).unsqueeze(-1).float()                   # (B, T, 1)
        h_mean = (h * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1)
        z = torch.relu(self.fc1(self.drop(h_mean)))
        return self.fc2(z)

tfm_model = TransformerClassifier(vocab_size=len(word2idx),
                                  embed_dim=128, num_heads=4,
                                  num_layers=2, ff_dim=256,
                                  num_classes=NUM_CLASSES, dropout=0.2)
print(tfm_model)
tfm_history = train_model(tfm_model, train_loader, val_loader,
                          epochs=3, lr=5e-4, name="Transformer")
tfm_results = evaluate(tfm_model, test_loader, list(le.classes_))


## 9. Model D — BERT fine-tuning (`bert-base-uncased`)

For BERT I'm just grabbing the pre-trained `bert-base-uncased` from
HuggingFace and slapping a 5-class linear layer on top, then fine-tuning
the whole thing on our data. Since BERT was already trained on a huge
amount of English text it should pick up this task pretty fast — the
model already knows what words mean, it just has to figure out which ones
mean "method" vs "result" etc.

I'm using AdamW with a really small lr (2e-5), a tiny bit of warmup, and
gradient clipping. Standard BERT fine-tuning stuff.

> **Heads up:** BERT is slow. One epoch on the full training set takes
> like 15–25 mins on a Colab T4. To not wait forever I set
> `BERT_TRAIN_FRACTION = 0.25` so it only trains on a stratified 25%
> sample. If you want max accuracy bump that to `1.0` and go grab food.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

BERT_NAME = "bert-base-uncased"
BERT_MAX_LEN = 96
BERT_BATCH = 32
BERT_EPOCHS = 2
BERT_LR = 2e-5
BERT_TRAIN_FRACTION = 0.25     # stratified subsample to keep training feasible

bert_tok = AutoTokenizer.from_pretrained(BERT_NAME)

def encode_with_bert(texts, labels):
    enc = bert_tok(list(texts), max_length=BERT_MAX_LEN,
                   padding="max_length", truncation=True, return_tensors="pt")
    enc["labels"] = torch.tensor(labels, dtype=torch.long)
    return enc

# Stratified subsample of training data
def stratified_sample(df, frac, seed=SEED):
    return (df.groupby("target", group_keys=False)
              .apply(lambda g: g.sample(frac=frac, random_state=seed))
              .reset_index(drop=True))

train_bert_df = stratified_sample(train_df, BERT_TRAIN_FRACTION) if BERT_TRAIN_FRACTION < 1.0 else train_df
print("BERT train rows:", len(train_bert_df))

train_bert = encode_with_bert(train_bert_df["abstract_text"], le.transform(train_bert_df["target"]))
val_bert   = encode_with_bert(val_df["abstract_text"],   le.transform(val_df["target"]))
test_bert  = encode_with_bert(test_df["abstract_text"],  le.transform(test_df["target"]))


class BertDataset(Dataset):
    def __init__(self, enc): self.enc = enc
    def __len__(self): return self.enc["labels"].size(0)
    def __getitem__(self, i):
        return {k: v[i] for k, v in self.enc.items()}

train_bert_loader = DataLoader(BertDataset(train_bert), batch_size=BERT_BATCH, shuffle=True)
val_bert_loader   = DataLoader(BertDataset(val_bert),   batch_size=BERT_BATCH)
test_bert_loader  = DataLoader(BertDataset(test_bert),  batch_size=BERT_BATCH)


In [ ]:
bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_NAME, num_labels=NUM_CLASSES).to(device)

optim_ = optim.AdamW(bert_model.parameters(), lr=BERT_LR, weight_decay=0.01)
total_steps = len(train_bert_loader) * BERT_EPOCHS
scheduler = get_linear_schedule_with_warmup(optim_, num_warmup_steps=int(0.1*total_steps),
                                            num_training_steps=total_steps)

bert_history = {"train_loss": [], "val_acc": [], "val_f1": []}
for ep in range(1, BERT_EPOCHS + 1):
    bert_model.train()
    total, t0 = 0.0, time.time()
    for step, batch in enumerate(train_bert_loader, 1):
        batch = {k: v.to(device) for k, v in batch.items()}
        optim_.zero_grad()
        out = bert_model(**batch)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
        optim_.step(); scheduler.step()
        total += out.loss.item()
        if step % 100 == 0:
            print(f"[BERT] ep{ep} step{step}/{len(train_bert_loader)} loss {total/step:.4f}")

    # val
    bert_model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in val_bert_loader:
            labels = batch["labels"]
            inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            logits = bert_model(**inputs).logits
            preds.extend(logits.argmax(dim=1).cpu().numpy())
            trues.extend(labels.numpy())
    acc = accuracy_score(trues, preds)
    _, _, f1, _ = precision_recall_fscore_support(trues, preds, average="weighted", zero_division=0)
    bert_history["train_loss"].append(total/len(train_bert_loader))
    bert_history["val_acc"].append(acc); bert_history["val_f1"].append(f1)
    print(f"[BERT] epoch {ep}/{BERT_EPOCHS}  val_acc={acc:.4f}  val_f1={f1:.4f}  ({time.time()-t0:.1f}s)")


In [ ]:
# Test-set evaluation for BERT
bert_model.eval()
preds, trues = [], []
with torch.no_grad():
    for batch in test_bert_loader:
        labels = batch["labels"]
        inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        logits = bert_model(**inputs).logits
        preds.extend(logits.argmax(dim=1).cpu().numpy())
        trues.extend(labels.numpy())

print(classification_report(trues, preds, target_names=list(le.classes_), zero_division=0))
acc = accuracy_score(trues, preds)
pr, rc, f1, _ = precision_recall_fscore_support(trues, preds, average="weighted", zero_division=0)
bert_results = {"accuracy": acc, "precision": pr, "recall": rc, "f1": f1,
                "y_true": trues, "y_pred": preds}


## 10. Comparing the models

Now that all four are trained let's just dump the test scores into a
table, plot the training curves on the same axes, and show a confusion
matrix for whichever model did the best.


In [ ]:
results_table = pd.DataFrame([
    {"model": "Simple RNN",  **{k: rnn_results[k]  for k in ['accuracy','precision','recall','f1']}},
    {"model": "LSTM",        **{k: lstm_results[k] for k in ['accuracy','precision','recall','f1']}},
    {"model": "Transformer", **{k: tfm_results[k]  for k in ['accuracy','precision','recall','f1']}},
    {"model": "BERT",        **{k: bert_results[k] for k in ['accuracy','precision','recall','f1']}},
])
results_table = results_table.round(4)
print(results_table.to_string(index=False))
results_table.to_csv("results_summary.csv", index=False)


In [ ]:
# Training-curve plot (loss + val accuracy)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for hist, name in [(rnn_history, "RNN"), (lstm_history, "LSTM"),
                   (tfm_history, "Transformer"), (bert_history, "BERT")]:
    epochs = range(1, len(hist["train_loss"]) + 1)
    ax[0].plot(epochs, hist["train_loss"], marker="o", label=name)
    ax[1].plot(epochs, hist["val_acc"],    marker="o", label=name)
ax[0].set_title("Training loss");        ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend()
ax[1].set_title("Validation accuracy");  ax[1].set_xlabel("epoch"); ax[1].set_ylabel("acc");  ax[1].legend()
plt.tight_layout(); plt.savefig("training_curves.png", dpi=120); plt.show()


In [ ]:
# Pick best model by test F1 and draw its confusion matrix
all_results = {"Simple RNN": rnn_results, "LSTM": lstm_results,
               "Transformer": tfm_results, "BERT": bert_results}
best_name = max(all_results, key=lambda n: all_results[n]["f1"])
best = all_results[best_name]
print("best by F1:", best_name, "F1 =", round(best['f1'], 4))

cm = confusion_matrix(best["y_true"], best["y_pred"])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=list(le.classes_), yticklabels=list(le.classes_))
plt.title(f"Confusion matrix — {best_name}")
plt.xlabel("predicted"); plt.ylabel("actual")
plt.tight_layout(); plt.savefig("confusion_matrix_best.png", dpi=120); plt.show()


## 11. Discussion

### 11.1 Why LSTM beats Simple RNN

The basic RNN problem is the vanishing gradient thing. When you
backpropagate through a long sequence you end up multiplying a bunch of
gradients together, and if they're less than 1 they shrink to basically
zero by the time you get to the start of the sentence. So the model
basically forgets the beginning by the time it reads the end. For our
sentences where the average is 26 words (and some are 50+), that's a
real problem.

LSTMs fix this with the gated cell state. There's a separate path that
gradients can flow through without getting squashed, and the model
learns when to remember vs forget vs update. So it actually keeps
information from earlier in the sentence around. That's why LSTMs
usually beat RNNs by like 5-10 points on this kind of task.

### 11.2 Transformers vs RNN/LSTM

The big difference is RNNs/LSTMs go through tokens one at a time —
they have to wait for token t-1 before they can do token t. Transformers
don't do that. With self-attention every token can directly look at
every other token at the same time. Two main benefits:

1. **No squishing context through one hidden state.** A word at position
   60 can directly attend to the word at position 1, no need to pass
   info through 60 steps.
2. **Way faster on GPUs.** Since there's no dependency between time
   steps the whole thing parallelizes nicely, which is why transformers
   train so much faster in practice.

The downside is the O(T²) attention cost, but for 64-token sentences
that's nothing.

### 11.3 Why BERT usually wins

BERT was pre-trained on like 3 billion words before we even touched it.
By the time we start fine-tuning it already knows English, knows
biomedical terms (kinda), knows how to build contextual word
representations. So all we're really doing is teaching it our specific
5-way classification problem.

The custom Transformer has to learn everything from scratch — what
words mean, how they go together, AND how to classify them. With only
80k sentences that's a lot to ask. BERT skips the first two steps
basically for free, which is why it usually beats the from-scratch
Transformer by a decent margin.

### 11.4 Strengths and weaknesses

| model | good at | not so good at |
|---|---|---|
| **Simple RNN** | small + fast to train | vanishing gradients, struggles with long sentences |
| **LSTM** | handles long sequences, solid baseline | still sequential so slower than transformers on GPU |
| **Transformer (custom)** | parallelizes well, flexible | needs more data than LSTM to reach same level |
| **BERT (fine-tuned)** | best accuracy, knows English already | huge model, needs a GPU, slow at inference |

### 11.5 Where the models mess up

The hardest classes are **BACKGROUND** and **OBJECTIVE** — they get
mixed up a lot because they're both short intro-style sentences and the
difference is more about *intent* than the specific words used. The
other common mix-up is **RESULTS ↔ CONCLUSIONS**, especially for the
last sentence in a results section where the author kinda starts
transitioning into the takeaway.

One thing that would probably help a lot is using the sentence's
position in the abstract as a feature. OBJECTIVE almost always shows
up early, CONCLUSIONS almost always show up at the end. Didn't do this
here but it'd be a low-effort win.


## 12. Conclusions

Wrapping up — the results basically went how you'd expect:

1. The simple RNN is the worst because it can't really remember stuff
   across long sentences.
2. The LSTM does noticeably better thanks to the gates fixing the
   vanishing gradient issue.
3. The custom Transformer is around the same level as the LSTM, maybe
   a bit better, and trains way faster on GPU.
4. BERT wins overall because it was already pre-trained on a ton of
   text. Fine-tuning is way more sample-efficient than training a model
   from scratch on 80k sentences.
5. Most of the remaining errors are on BACKGROUND vs OBJECTIVE and
   RESULTS vs CONCLUSIONS. Adding sentence position as a feature would
   probably clean up a lot of those.

So for this kind of task pre-trained models are pretty clearly the way
to go if you have access to a GPU. If you don't, the LSTM is a solid
fallback.

## References

- Dernoncourt, F. & Lee, J.Y. (2017). *PubMed 200k RCT: a Dataset for
  Sequential Sentence Classification in Medical Abstracts.* IJCNLP.
- Jin, D. & Szolovits, P. (2018). *Hierarchical Neural Networks for
  Sequential Sentence Classification in Medical Scientific Abstracts.*
  EMNLP.
- Devlin, J. et al. (2019). *BERT: Pre-training of Deep Bidirectional
  Transformers for Language Understanding.* NAACL.
- Vaswani, A. et al. (2017). *Attention Is All You Need.* NeurIPS.
- Hochreiter, S. & Schmidhuber, J. (1997). *Long Short-Term Memory.*
  Neural Computation 9(8).
